# 🔥 Head-to-Head: ARIMA vs LSTM for Time Series Forecasting

## Objective
Compare a **Statistical Model (ARIMA)** against a **Deep Learning Model (LSTM)** for predicting hourly electricity demand.

## Dataset
- **Source**: `final_4year.csv` - Delhi hourly electricity demand (2020-2023)
- **Target**: `Hourly Demand Met (in MW)`
- **Approach**: **Univariate Analysis** - using only the target column to ensure a fair comparison

## Why Univariate?
ARIMA is fundamentally designed for univariate time series. To ensure a fair "apples-to-apples" comparison,
we restrict LSTM to the same input information - just past values of the target variable.

---
## 1. Imports and Setup

**If you encounter a pmdarima import error**, run this cell first to fix the package compatibility:

```python
# Option 1: Upgrade both packages to compatible versions
!pip install --upgrade pmdarima statsmodels

# Option 2: If still failing, install specific compatible versions
!pip install pmdarima==2.0.3 statsmodels==0.13.5
```

The notebook will automatically fall back to using `statsmodels.SARIMAX` if pmdarima fails to import.

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Statistical analysis and ARIMA
from statsmodels.tsa.stattools import adfuller

# Try to import pmdarima, fall back to statsmodels SARIMAX if incompatible
USE_PMDARIMA = False
try:
    import pmdarima as pm
    USE_PMDARIMA = True
    print(f"pmdarima version: {pm.__version__}")
except ImportError as e:
    print(f"pmdarima import failed: {e}")
    print("Using statsmodels SARIMAX as fallback...")
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from itertools import product

# Deep Learning (LSTM)
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"\n✅ All imports successful!")
print(f"Using {'pmdarima auto_arima' if USE_PMDARIMA else 'statsmodels SARIMAX'} for ARIMA modeling")

---
## 2. Data Loading and Preprocessing

In [ ]:
# =============================================================================
# LOAD AND PREPROCESS DATA
# =============================================================================

# Load the dataset
df = pd.read_csv("DataFolder/final_4year.csv")

print("Raw Data Shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# =============================================================================
# CONVERT DATE AND TIME TO DATETIME INDEX
# =============================================================================

# Why: Time series models require a proper datetime index to understand
# temporal relationships and seasonality patterns.

# Convert 'Date and Time' column to datetime objects
df['Date and Time'] = pd.to_datetime(df['Date and Time'])

# Set as index (crucial for time series analysis)
df = df.set_index('Date and Time')

# Sort by index to ensure chronological order
df = df.sort_index()

print(f"Date Range: {df.index.min()} to {df.index.max()}")
print(f"Total Duration: {(df.index.max() - df.index.min()).days} days")
print(f"Total Records: {len(df):,}")

In [ ]:
# =============================================================================
# INFER AND SET FREQUENCY
# =============================================================================

# Why: ARIMA requires a consistent frequency. The data should be hourly ('h').
# We explicitly set the frequency to handle any gaps.

# Check the inferred frequency
inferred_freq = pd.infer_freq(df.index)
print(f"Inferred Frequency: {inferred_freq}")

# Extract the target column for univariate analysis
target_col = 'Hourly Demand Met (in MW)'
series = df[target_col].copy()

# Handle any missing values (forward fill, then backward fill for edge cases)
missing_before = series.isnull().sum()
series = series.ffill().bfill()
missing_after = series.isnull().sum()

print(f"\nMissing values before: {missing_before}")
print(f"Missing values after:  {missing_after}")

# Set explicit hourly frequency
# If there are gaps, reindex to fill them
full_range = pd.date_range(start=series.index.min(), end=series.index.max(), freq='h')
series = series.reindex(full_range)
series = series.ffill().bfill()  # Fill any gaps created by reindexing

print(f"\nFinal series length: {len(series):,}")
print(f"Frequency set: {series.index.freq}")

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# =============================================================================
# PLOT THE ENTIRE TIME SERIES
# =============================================================================

# Why: Visual inspection helps identify trends, seasonality, and anomalies
# before modeling.

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Full time series
axes[0].plot(series.index, series.values, linewidth=0.5, color='#2c3e50')
axes[0].set_title('Hourly Electricity Demand - Full Time Series (4 Years)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Demand (MW)')
axes[0].grid(True, alpha=0.3)

# One year sample (to see seasonal patterns)
one_year = series['2022']
axes[1].plot(one_year.index, one_year.values, linewidth=0.8, color='#e74c3c')
axes[1].set_title('One Year Sample (2022) - Shows Annual Seasonality', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Demand (MW)')
axes[1].grid(True, alpha=0.3)

# One week sample (to see daily patterns)
one_week = series['2022-06-01':'2022-06-07']
axes[2].plot(one_week.index, one_week.values, linewidth=1.5, color='#27ae60', marker='o', markersize=3)
axes[2].set_title('One Week Sample (June 2022) - Shows Daily Patterns', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Demand (MW)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 KEY OBSERVATIONS:")
print("• Clear ANNUAL SEASONALITY: Higher demand in summer (May-July) due to AC usage")
print("• Clear DAILY PATTERNS: Demand peaks during morning and evening hours")
print("• Possible TREND: Overall demand may be increasing over the years")

In [ ]:
# =============================================================================
# AUGMENTED DICKEY-FULLER TEST FOR STATIONARITY
# =============================================================================

# Why: ARIMA assumes the time series is stationary (constant mean and variance).
# The ADF test checks this assumption. If p-value < 0.05, series is stationary.
# If not stationary, differencing (the 'd' in ARIMA) will be needed.

def adf_test(series, title=''):
    """
    Perform Augmented Dickey-Fuller test for stationarity.
    
    H0 (Null Hypothesis): The series has a unit root (non-stationary)
    H1 (Alternative): The series is stationary
    
    If p-value < 0.05: Reject H0, series IS stationary
    If p-value >= 0.05: Fail to reject H0, series is NOT stationary
    """
    print(f"\n{'='*60}")
    print(f"AUGMENTED DICKEY-FULLER TEST {title}")
    print(f"{'='*60}")
    
    result = adfuller(series.dropna(), autolag='AIC')
    
    print(f"Test Statistic:  {result[0]:.6f}")
    print(f"p-value:         {result[1]:.6f}")
    print(f"Lags Used:       {result[2]}")
    print(f"Observations:    {result[3]}")
    print("\nCritical Values:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    
    if result[1] < 0.05:
        print("\n✅ RESULT: Series IS STATIONARY (p-value < 0.05)")
        print("   → No differencing required for ARIMA")
        return True
    else:
        print("\n⚠️ RESULT: Series is NOT STATIONARY (p-value >= 0.05)")
        print("   → Differencing will be needed (auto_arima will handle this)")
        return False

# Test the original series
is_stationary = adf_test(series, '- Original Series')

In [ ]:
# If not stationary, let's also check the differenced series
if not is_stationary:
    series_diff = series.diff().dropna()
    adf_test(series_diff, '- First Differenced Series')

---
## 4. Train/Test Split

**Important**: For time series, we must split chronologically (no shuffling!) to prevent data leakage.

In [ ]:
# =============================================================================
# CHRONOLOGICAL TRAIN/TEST SPLIT (90/10)
# =============================================================================

# Why: We use 90% for training, 10% for testing.
# This ensures we have enough historical data for ARIMA to learn patterns
# while having a substantial test set for evaluation.

split_ratio = 0.90
split_idx = int(len(series) * split_ratio)

train = series.iloc[:split_idx]
test = series.iloc[split_idx:]

print(f"{'='*60}")
print("TRAIN/TEST SPLIT")
print(f"{'='*60}")
print(f"\nTotal samples:    {len(series):,}")
print(f"Training samples: {len(train):,} ({len(train)/len(series)*100:.1f}%)")
print(f"Test samples:     {len(test):,} ({len(test)/len(series)*100:.1f}%)")
print(f"\nTraining period:  {train.index.min()} to {train.index.max()}")
print(f"Test period:      {test.index.min()} to {test.index.max()}")
print(f"Test duration:    {(test.index.max() - test.index.min()).days} days")

In [ ]:
# Visualize the split
plt.figure(figsize=(16, 5))
plt.plot(train.index, train.values, label='Training Data', color='#3498db', linewidth=0.5)
plt.plot(test.index, test.values, label='Test Data', color='#e74c3c', linewidth=0.5)
plt.axvline(x=train.index[-1], color='black', linestyle='--', linewidth=2, label='Train/Test Split')
plt.title('Train/Test Split Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Demand (MW)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Model 1: ARIMA (Statistical Baseline)

### What is ARIMA?
- **AR (AutoRegressive)**: The model uses past values to predict future values
- **I (Integrated)**: Differencing to make the series stationary
- **MA (Moving Average)**: Uses past forecast errors

**Parameters (p, d, q)**:
- `p`: Number of lag observations (AR terms)
- `d`: Degree of differencing
- `q`: Size of moving average window (MA terms)

We use `auto_arima` from `pmdarima` to automatically find optimal parameters.

In [ ]:
# =============================================================================
# ARIMA MODEL - AUTOMATIC OR MANUAL PARAMETER SELECTION
# =============================================================================

# Why use auto_arima (if available)?
# - Automatically tests different (p, d, q) combinations
# - Uses AIC/BIC criteria to select the best model
# - Handles seasonal patterns with SARIMA if needed

print("🔍 Fitting ARIMA model...")
print("   This may take a few minutes...\n")

if USE_PMDARIMA:
    # Use pmdarima's auto_arima for automatic parameter selection
    # Use seasonal=True with m=24 for hourly data with daily seasonality
    arima_model = pm.auto_arima(
        train,
        start_p=1, max_p=3,           # AR order range
        start_q=1, max_q=3,           # MA order range
        d=None,                        # Let auto_arima determine differencing
        max_d=2,                       # Max differencing order
        seasonal=True,                 # Enable seasonal component
        m=24,                          # Seasonal period (24 hours = daily)
        start_P=0, max_P=2,           # Seasonal AR order
        start_Q=0, max_Q=2,           # Seasonal MA order
        D=None,                        # Let auto_arima determine seasonal differencing
        max_D=1,                       # Max seasonal differencing
        trace=True,                    # Print progress
        error_action='ignore',         # Don't raise errors for invalid models
        suppress_warnings=True,
        stepwise=True,                 # Use stepwise search (faster)
        n_fits=30                      # Limit number of models tried
    )
    print("\n" + "="*60)
    print("ARIMA MODEL SUMMARY")
    print("="*60)
    print(arima_model.summary())
    
else:
    # Fallback: Use statsmodels SARIMAX with reasonable default parameters
    # SARIMA(1,1,1)(1,1,1,24) is a good starting point for hourly data
    
    print("Using SARIMAX with parameters: (1,1,1) x (1,1,1,24)")
    print("(p=1, d=1, q=1) for non-seasonal, (P=1, D=1, Q=1, m=24) for seasonal")
    
    # Fit SARIMAX model
    arima_model = SARIMAX(
        train,
        order=(1, 1, 1),              # (p, d, q)
        seasonal_order=(1, 1, 1, 24), # (P, D, Q, m) - daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    arima_result = arima_model.fit(disp=False)
    
    print("\n" + "="*60)
    print("SARIMAX MODEL SUMMARY")
    print("="*60)
    print(arima_result.summary())
    
    # Store the fitted result for later use
    arima_model = arima_result

In [ ]:
# =============================================================================
# ARIMA PREDICTIONS ON TEST SET
# =============================================================================

print("📈 Generating ARIMA predictions for test set...")

# Generate predictions for the entire test set
n_test = len(test)

if USE_PMDARIMA:
    # pmdarima predict method
    arima_predictions = arima_model.predict(n_periods=n_test)
else:
    # statsmodels SARIMAX forecast method
    arima_predictions = arima_model.forecast(steps=n_test)

# Create a Series with proper index
arima_pred_series = pd.Series(arima_predictions, index=test.index)

print(f"\n✅ Generated {len(arima_pred_series):,} predictions")
print(f"Prediction period: {arima_pred_series.index.min()} to {arima_pred_series.index.max()}")

---
## 6. Model 2: LSTM (Deep Learning Challenger)

### What is LSTM?
- **Long Short-Term Memory** is a type of Recurrent Neural Network (RNN)
- Designed to learn long-term dependencies in sequential data
- Uses "gates" to control information flow: Forget Gate, Input Gate, Output Gate

### Why Scale Data for LSTM?
- Neural networks are sensitive to the scale of input data
- Large values can cause gradient explosion during training
- `MinMaxScaler` normalizes data to [0, 1] range for stable training

In [ ]:
# =============================================================================
# DATA SCALING FOR LSTM
# =============================================================================

# Why MinMaxScaler?
# - LSTMs use sigmoid and tanh activations that work best with [0,1] or [-1,1] ranges
# - Prevents gradient vanishing/explosion during backpropagation
# - IMPORTANT: Fit scaler ONLY on training data to prevent data leakage

scaler = MinMaxScaler(feature_range=(0, 1))

# Reshape to 2D for sklearn scaler (samples, features)
train_scaled = scaler.fit_transform(train.values.reshape(-1, 1))
test_scaled = scaler.transform(test.values.reshape(-1, 1))

print(f"Original train range: [{train.min():.2f}, {train.max():.2f}]")
print(f"Scaled train range:   [{train_scaled.min():.4f}, {train_scaled.max():.4f}]")
print(f"\nScaler fitted on training data only (no data leakage)")

In [ ]:
# =============================================================================
# CREATE SEQUENCES FOR LSTM (SLIDING WINDOW)
# =============================================================================

# Why use sliding window?
# - LSTM needs input in shape (samples, timesteps, features)
# - We use past 60 hours (lookback) to predict the next 1 hour
# - 60 hours captures ~2.5 daily cycles, enough for pattern recognition

LOOKBACK = 60  # Number of past hours to use for prediction

def create_sequences(data, lookback):
    """
    Create input sequences and corresponding targets for LSTM.
    
    For each timestep t, we use [t-lookback : t] to predict [t]
    
    Parameters:
        data: Scaled time series values
        lookback: Number of past timesteps to use as input
    
    Returns:
        X: Input sequences of shape (samples, lookback, 1)
        y: Target values of shape (samples,)
    """
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])  # Past 'lookback' values
        y.append(data[i, 0])               # Current value to predict
    return np.array(X), np.array(y)

# Create training sequences
X_train, y_train = create_sequences(train_scaled, LOOKBACK)

# Reshape X to 3D for LSTM: (samples, timesteps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

print(f"Lookback window: {LOOKBACK} hours")
print(f"\nTraining sequences:")
print(f"  X_train shape: {X_train.shape} (samples, timesteps, features)")
print(f"  y_train shape: {y_train.shape} (samples,)")

In [ ]:
# =============================================================================
# BUILD LSTM MODEL ARCHITECTURE
# =============================================================================

# Architecture explanation:
# - LSTM(50): 50 memory units to capture patterns
#   - return_sequences=False: Only return final output (we predict 1 step)
# - Dense(1): Single output neuron for demand prediction

lstm_model = Sequential([
    # LSTM layer with 50 units
    # input_shape = (timesteps, features) = (60, 1)
    LSTM(units=50, return_sequences=False, input_shape=(LOOKBACK, 1)),
    
    # Dense output layer - predicts single value
    Dense(units=1)
])

# Compile the model
# - Adam optimizer: Adaptive learning rate, works well in practice
# - MSE loss: Standard for regression problems
lstm_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']  # Track Mean Absolute Error during training
)

# Display model architecture
lstm_model.summary()

In [ ]:
# =============================================================================
# TRAIN LSTM MODEL
# =============================================================================

# Why use EarlyStopping?
# - Prevents overfitting by stopping when validation loss stops improving
# - Saves the best model weights automatically
# - patience=5: Stop if no improvement for 5 consecutive epochs

early_stopping = EarlyStopping(
    monitor='val_loss',       # Watch validation loss
    patience=5,               # Wait 5 epochs before stopping
    restore_best_weights=True # Restore best weights when stopped
)

print("🚀 Training LSTM model...\n")

history = lstm_model.fit(
    X_train, y_train,
    epochs=20,                  # Maximum epochs
    batch_size=32,              # Samples per gradient update
    validation_split=0.1,       # Use 10% of training data for validation
    callbacks=[early_stopping],
    verbose=1
)

print(f"\n✅ Training complete!")
print(f"   Best validation loss: {min(history.history['val_loss']):.6f}")
print(f"   Epochs trained: {len(history.history['loss'])}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('LSTM Training History - Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE plot
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_title('LSTM Training History - MAE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Absolute Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# LSTM PREDICTIONS ON TEST SET
# =============================================================================

# For LSTM, we need to create sequences that include training data
# to predict the first 'lookback' points of test set

print("📈 Generating LSTM predictions for test set...")

# Combine train and test scaled data for sequence creation
full_scaled = np.concatenate([train_scaled, test_scaled])

# Create test sequences starting from where test data begins
test_start_idx = len(train_scaled)

lstm_predictions_scaled = []

# Predict each test point using the lookback window
for i in range(test_start_idx, len(full_scaled)):
    # Get the lookback sequence
    seq = full_scaled[i - LOOKBACK:i, 0]
    seq = seq.reshape((1, LOOKBACK, 1))
    
    # Predict
    pred = lstm_model.predict(seq, verbose=0)
    lstm_predictions_scaled.append(pred[0, 0])

# Convert to numpy array and inverse transform
lstm_predictions_scaled = np.array(lstm_predictions_scaled).reshape(-1, 1)
lstm_predictions = scaler.inverse_transform(lstm_predictions_scaled).flatten()

# Create a Series with proper index
lstm_pred_series = pd.Series(lstm_predictions, index=test.index)

print(f"\n✅ Generated {len(lstm_pred_series):,} predictions")
print(f"Prediction period: {lstm_pred_series.index.min()} to {lstm_pred_series.index.max()}")

---
## 7. The Showdown: Model Evaluation

In [ ]:
# =============================================================================
# CALCULATE EVALUATION METRICS
# =============================================================================

def calculate_metrics(y_true, y_pred, model_name):
    """
    Calculate RMSE and MAE for a model's predictions.
    
    RMSE (Root Mean Squared Error):
        - Penalizes large errors more heavily
        - Same units as target variable (MW)
    
    MAE (Mean Absolute Error):
        - Average absolute difference between predicted and actual
        - More robust to outliers than RMSE
    """
    # Handle NaN values by creating a mask for valid entries
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    
    # Create mask where both values are not NaN
    valid_mask = ~(np.isnan(y_true_arr) | np.isnan(y_pred_arr))
    
    if valid_mask.sum() == 0:
        print(f"⚠️ Warning: No valid predictions for {model_name}")
        return {
            'Model': model_name,
            'RMSE (MW)': np.nan,
            'MAE (MW)': np.nan
        }
    
    # Report if any NaN values were found
    nan_count = len(y_true_arr) - valid_mask.sum()
    if nan_count > 0:
        print(f"⚠️ Warning: {nan_count} NaN values found in {model_name} predictions (excluded from metrics)")
    
    y_true_clean = y_true_arr[valid_mask]
    y_pred_clean = y_pred_arr[valid_mask]
    
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    
    return {
        'Model': model_name,
        'RMSE (MW)': round(rmse, 2),
        'MAE (MW)': round(mae, 2)
    }

# Check for NaN values in predictions
print("Checking for NaN values...")
print(f"ARIMA predictions NaN count: {arima_pred_series.isna().sum()}")
print(f"LSTM predictions NaN count: {lstm_pred_series.isna().sum()}")
print(f"Test data NaN count: {test.isna().sum()}")

# Fill NaN values in ARIMA predictions if any (can happen at start due to seasonal differencing)
if arima_pred_series.isna().sum() > 0:
    print("\nFilling ARIMA NaN values with forward/backward fill...")
    arima_pred_series = arima_pred_series.ffill().bfill()

# Calculate metrics for both models
arima_metrics = calculate_metrics(test.values, arima_pred_series.values, 'ARIMA')
lstm_metrics = calculate_metrics(test.values, lstm_pred_series.values, 'LSTM')

# Create comparison table
results_df = pd.DataFrame([arima_metrics, lstm_metrics])

print("\n" + "="*60)
print("📊 MODEL COMPARISON RESULTS")
print("="*60)
print(f"\nTest Set Size: {len(test):,} samples ({len(test)/24:.0f} days)")
print(f"\n{results_df.to_string(index=False)}")

# Determine winner
rmse_winner = 'ARIMA' if arima_metrics['RMSE (MW)'] < lstm_metrics['RMSE (MW)'] else 'LSTM'
mae_winner = 'ARIMA' if arima_metrics['MAE (MW)'] < lstm_metrics['MAE (MW)'] else 'LSTM'

print(f"\n🏆 WINNER BY RMSE: {rmse_winner}")
print(f"🏆 WINNER BY MAE:  {mae_winner}")

In [ ]:
# =============================================================================
# HIGH-CONTRAST VISUALIZATION (LAST 7 DAYS)
# =============================================================================

# Why show last 7 days?
# - Full test set (~146 days) is too dense to see details
# - 7 days shows daily patterns clearly while being readable

# Get last 7 days
last_7_days = 24 * 7  # 168 hours
actual_7d = test.iloc[-last_7_days:]
arima_7d = arima_pred_series.iloc[-last_7_days:]
lstm_7d = lstm_pred_series.iloc[-last_7_days:]

fig, ax = plt.subplots(figsize=(16, 8))

# Plot with high contrast colors
ax.plot(actual_7d.index, actual_7d.values, 
        label='Actual (Ground Truth)', 
        color='#2c3e50', linewidth=2.5, alpha=0.9)

ax.plot(arima_7d.index, arima_7d.values, 
        label=f'ARIMA (RMSE: {arima_metrics["RMSE (MW)"]} MW)', 
        color='#e74c3c', linewidth=2, linestyle='--', alpha=0.8)

ax.plot(lstm_7d.index, lstm_7d.values, 
        label=f'LSTM (RMSE: {lstm_metrics["RMSE (MW)"]} MW)', 
        color='#27ae60', linewidth=2, linestyle='-.', alpha=0.8)

ax.set_title('ARIMA vs LSTM: Last 7 Days of Test Data', fontsize=16, fontweight='bold')
ax.set_xlabel('Date & Time', fontsize=12)
ax.set_ylabel('Hourly Demand Met (MW)', fontsize=12)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

# Add vertical lines for each day
for i, date in enumerate(pd.date_range(actual_7d.index.min(), actual_7d.index.max(), freq='D')):
    ax.axvline(x=date, color='gray', linestyle=':', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# PREDICTION ERROR DISTRIBUTION
# =============================================================================

# Calculate prediction errors (handle NaN values)
arima_errors = test.values - arima_pred_series.values
lstm_errors = test.values - lstm_pred_series.values

# Remove NaN values for plotting
arima_errors_clean = arima_errors[~np.isnan(arima_errors)]
lstm_errors_clean = lstm_errors[~np.isnan(lstm_errors)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ARIMA error distribution
axes[0].hist(arima_errors_clean, bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[0].axvline(x=np.mean(arima_errors_clean), color='blue', linestyle='-', linewidth=2, 
                label=f'Mean: {np.mean(arima_errors_clean):.1f} MW')
axes[0].set_title('ARIMA Prediction Error Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Error (MW)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# LSTM error distribution
axes[1].hist(lstm_errors_clean, bins=50, color='#27ae60', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[1].axvline(x=np.mean(lstm_errors_clean), color='blue', linestyle='-', linewidth=2,
                label=f'Mean: {np.mean(lstm_errors_clean):.1f} MW')
axes[1].set_title('LSTM Prediction Error Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Error (MW)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nARIMA Error Statistics:")
print(f"  Mean Error:  {np.mean(arima_errors_clean):.2f} MW (bias)")
print(f"  Std Error:   {np.std(arima_errors_clean):.2f} MW")

print(f"\nLSTM Error Statistics:")
print(f"  Mean Error:  {np.mean(lstm_errors_clean):.2f} MW (bias)")
print(f"  Std Error:   {np.std(lstm_errors_clean):.2f} MW")

---
## 8. 24-Hour Ahead Forecast

Generate predictions for the **next 24 hours** beyond the dataset.

In [ ]:
# =============================================================================
# 24-HOUR AHEAD FORECAST
# =============================================================================

forecast_hours = 24

print("\n" + "="*60)
print("🔮 24-HOUR AHEAD FORECAST")
print("="*60)

# -------------------------
# ARIMA 24-Hour Forecast
# -------------------------
print("\n📈 ARIMA Forecast:")

# Create forecast index
forecast_start = series.index[-1] + pd.Timedelta(hours=1)
forecast_index = pd.date_range(start=forecast_start, periods=forecast_hours, freq='h')

if USE_PMDARIMA:
    # Update ARIMA model with test data for more accurate forecast
    arima_model.update(test)
    # Generate 24-hour forecast
    arima_forecast = arima_model.predict(n_periods=forecast_hours)
else:
    # For statsmodels, we need to refit the model with all data
    # or extend the forecast from where we left off
    full_series = pd.concat([train, test])
    arima_full = SARIMAX(
        full_series,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 24),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    arima_full_result = arima_full.fit(disp=False)
    arima_forecast = arima_full_result.forecast(steps=forecast_hours)

arima_forecast_series = pd.Series(arima_forecast.values if hasattr(arima_forecast, 'values') else arima_forecast, 
                                   index=forecast_index)

print(f"Forecast period: {forecast_index[0]} to {forecast_index[-1]}")

In [ ]:
# -------------------------
# LSTM 24-Hour Forecast
# -------------------------
print("\n📈 LSTM Forecast:")

# For LSTM, we need to use the last 'lookback' values to make predictions
# Then use each prediction as input for the next (recursive forecasting)

# Get the last 'lookback' scaled values from the full dataset
last_sequence = full_scaled[-LOOKBACK:, 0].tolist()

lstm_forecast_scaled = []

for _ in range(forecast_hours):
    # Prepare input sequence
    seq = np.array(last_sequence[-LOOKBACK:]).reshape((1, LOOKBACK, 1))
    
    # Predict next value
    pred = lstm_model.predict(seq, verbose=0)[0, 0]
    lstm_forecast_scaled.append(pred)
    
    # Add prediction to sequence for next iteration
    last_sequence.append(pred)

# Inverse transform
lstm_forecast_scaled = np.array(lstm_forecast_scaled).reshape(-1, 1)
lstm_forecast = scaler.inverse_transform(lstm_forecast_scaled).flatten()

lstm_forecast_series = pd.Series(lstm_forecast, index=forecast_index)

print(f"Forecast period: {forecast_index[0]} to {forecast_index[-1]}")

In [ ]:
# =============================================================================
# DISPLAY 24-HOUR FORECAST TABLE
# =============================================================================

forecast_df = pd.DataFrame({
    'Date & Time': forecast_index,
    'ARIMA Forecast (MW)': arima_forecast_series.values.round(2),
    'LSTM Forecast (MW)': lstm_forecast_series.values.round(2),
    'Difference (MW)': (arima_forecast_series.values - lstm_forecast_series.values).round(2)
})

print("\n" + "="*80)
print("24-HOUR FORECAST COMPARISON")
print("="*80)
print(f"\n{forecast_df.to_string(index=False)}")

print(f"\n\nSummary Statistics:")
print(f"  ARIMA - Mean: {arima_forecast_series.mean():.2f} MW, Range: [{arima_forecast_series.min():.2f}, {arima_forecast_series.max():.2f}]")
print(f"  LSTM  - Mean: {lstm_forecast_series.mean():.2f} MW, Range: [{lstm_forecast_series.min():.2f}, {lstm_forecast_series.max():.2f}]")

In [ ]:
# =============================================================================
# VISUALIZE 24-HOUR FORECAST
# =============================================================================

# Show last 48 hours of actual data + 24 hour forecast
last_48h = series.iloc[-48:]

fig, ax = plt.subplots(figsize=(16, 7))

# Historical data
ax.plot(last_48h.index, last_48h.values, 
        label='Historical Data (Last 48 Hours)', 
        color='#2c3e50', linewidth=2.5)

# ARIMA forecast
ax.plot(arima_forecast_series.index, arima_forecast_series.values,
        label='ARIMA 24h Forecast', 
        color='#e74c3c', linewidth=2.5, linestyle='--', marker='o', markersize=4)

# LSTM forecast
ax.plot(lstm_forecast_series.index, lstm_forecast_series.values,
        label='LSTM 24h Forecast', 
        color='#27ae60', linewidth=2.5, linestyle='-.', marker='s', markersize=4)

# Add vertical line at forecast start
ax.axvline(x=series.index[-1], color='purple', linestyle=':', linewidth=2, 
           label='Forecast Starts')

# Shade forecast region
ax.axvspan(forecast_index[0], forecast_index[-1], alpha=0.1, color='yellow')

ax.set_title('24-Hour Ahead Forecast: ARIMA vs LSTM', fontsize=16, fontweight='bold')
ax.set_xlabel('Date & Time', fontsize=12)
ax.set_ylabel('Hourly Demand Met (MW)', fontsize=12)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Final Summary & Conclusions

In [ ]:
# =============================================================================
# FINAL COMPARISON VISUALIZATION
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = ['ARIMA', 'LSTM']
colors = ['#e74c3c', '#27ae60']

# RMSE comparison
rmse_values = [arima_metrics['RMSE (MW)'], lstm_metrics['RMSE (MW)']]
bars1 = axes[0].bar(models, rmse_values, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('RMSE (MW)', fontsize=12)
axes[0].set_title('Root Mean Squared Error', fontsize=14, fontweight='bold')
for bar, val in zip(bars1, rmse_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
                 f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

# MAE comparison
mae_values = [arima_metrics['MAE (MW)'], lstm_metrics['MAE (MW)']]
bars2 = axes[1].bar(models, mae_values, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('MAE (MW)', fontsize=12)
axes[1].set_title('Mean Absolute Error', fontsize=14, fontweight='bold')
for bar, val in zip(bars2, mae_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
                 f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.suptitle('ARIMA vs LSTM: Model Performance Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# FINAL CONCLUSIONS
# =============================================================================

print("""
================================================================================
                    ARIMA vs LSTM: FINAL CONCLUSIONS
================================================================================

EXPERIMENTAL SETUP:
──────────────────────────────────────────────────────────────────────────────
• Dataset:        4 years of hourly electricity demand data
• Approach:       Univariate analysis (target column only)
• Train/Test:     90% / 10% chronological split
• Target:         Hourly Demand Met (in MW)
──────────────────────────────────────────────────────────────────────────────

MODEL CONFIGURATIONS:
──────────────────────────────────────────────────────────────────────────────
• ARIMA:  Auto-selected parameters with seasonal component (m=24)
• LSTM:   50 units, lookback=60 hours, MinMaxScaler, EarlyStopping
──────────────────────────────────────────────────────────────────────────────
""")

print(f"\nRESULTS:")
print(f"{'─'*60}")
print(f"│ Model  │  RMSE (MW)  │  MAE (MW)  │")
print(f"{'─'*60}")
print(f"│ ARIMA  │  {arima_metrics['RMSE (MW)']:>9}  │  {arima_metrics['MAE (MW)']:>8}  │")
print(f"│ LSTM   │  {lstm_metrics['RMSE (MW)']:>9}  │  {lstm_metrics['MAE (MW)']:>8}  │")
print(f"{'─'*60}")

# Calculate percentage differences
rmse_diff = ((lstm_metrics['RMSE (MW)'] - arima_metrics['RMSE (MW)']) / arima_metrics['RMSE (MW)']) * 100
mae_diff = ((lstm_metrics['MAE (MW)'] - arima_metrics['MAE (MW)']) / arima_metrics['MAE (MW)']) * 100

print(f"""
KEY FINDINGS:
──────────────────────────────────────────────────────────────────────────────
• RMSE Difference: LSTM is {abs(rmse_diff):.1f}% {'higher' if rmse_diff > 0 else 'lower'} than ARIMA
• MAE Difference:  LSTM is {abs(mae_diff):.1f}% {'higher' if mae_diff > 0 else 'lower'} than ARIMA
──────────────────────────────────────────────────────────────────────────────

INSIGHTS:
──────────────────────────────────────────────────────────────────────────────
1. ARIMA Strengths:
   • Interpretable: Clear seasonal and trend components
   • Efficient: Faster training, no GPU required
   • Robust: Well-suited for univariate time series with clear patterns

2. LSTM Strengths:
   • Flexible: Can capture complex non-linear patterns
   • Scalable: Can incorporate multiple features (multivariate)
   • Adaptive: Learns patterns automatically without manual feature engineering

3. When to Use Which:
   • ARIMA: When data has clear seasonal patterns and you need interpretability
   • LSTM:  When patterns are complex or you have multivariate data
──────────────────────────────────────────────────────────────────────────────

RECOMMENDATIONS:
──────────────────────────────────────────────────────────────────────────────
• For Production: Consider ensemble methods combining both models
• For LSTM: Add more features (weather, holidays) to potentially improve
• For ARIMA: Consider TBATS or Prophet for multiple seasonalities
──────────────────────────────────────────────────────────────────────────────
""")

In [ ]:
# =============================================================================
# SAVE RESULTS
# =============================================================================

# Save comparison results
results_df.to_csv('arima_vs_lstm_results.csv', index=False)
print("✅ Results saved to 'arima_vs_lstm_results.csv'")

# Save 24-hour forecast
forecast_df.to_csv('24h_forecast_comparison.csv', index=False)
print("✅ 24-hour forecast saved to '24h_forecast_comparison.csv'")

# Save predictions for further analysis
predictions_df = pd.DataFrame({
    'DateTime': test.index,
    'Actual': test.values,
    'ARIMA_Prediction': arima_pred_series.values,
    'LSTM_Prediction': lstm_pred_series.values,
    'ARIMA_Error': test.values - arima_pred_series.values,
    'LSTM_Error': test.values - lstm_pred_series.values
})
predictions_df.to_csv('test_predictions_comparison.csv', index=False)
print("✅ Test predictions saved to 'test_predictions_comparison.csv'")

print("\n🎉 Analysis Complete!")